# Maximum mixed layer depth in ACCESS-OM3

Related issue: https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38

Requires access to `/g/data/av17`

In [ ]:
# These first two cells must be in all notebooks!
# It allows us to run all the notebooks at once, this cell has a tag "parameters" which allows us to pass in 
# arguments externally using papermill (see mkfigs.sh for details)

# Set esm_file to the datastore for the main experiment of interest
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf+wombatlite-test3v2-00532b88/datastore.json"

# papermill settings. *No need to modify these if running interactively.* 
papermill = False                      # `cwd` and `nbname` will be populated by papermill.
cwd = None                             # current working directory 
nbname = None                          # notebook name

In [ ]:
# Parameters
esm_file = "/g/data/ol01/outputs/access-om3-25km/MC_25km_jra_iaf-1.0-beta-gm4-9fd08880/datastore.json"
papermill = True
cwd = "/g/data/tm70/cyb561/access-om3-paper-1-runs/MC_25km_jra_iaf-1.0-beta-gm4-9fd08880/notebooks/mkfigs_output_MC_25km_jra_iaf-1.0-beta-gm4-9fd08880/"
nbname = "MLD_max.ipynb"


In [ ]:
import os
if not papermill: 
    import nci_ipynb  # requires conda/analysis3-26.03 or later
    cwd = nci_ipynb.dir()
    nbname = nci_ipynb.name()
    os.chdir(cwd)
import mkfigs_bootstrap  # noqa: adds external/access-model-mkfigs/src to sys.path (stop-gap)
from mkfigs import MkmdWriter
mkmd = MkmdWriter(esm_file, nbname, str(cwd), pm=papermill)

In [ ]:
import xarray as xr
import cf_xarray as cfxr
import cf_xarray.units
import pint_xarray
from pint import application_registry as ureg
import intake
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
from distributed import Client
import cftime
import os
import matplotlib.pyplot as plt
import cmocean as cm
import cartopy.crs as ccrs
import cartopy.feature as cft
from textwrap import wrap
xr.set_options(keep_attrs=True);  # cf_xarray works best when xarray keeps attributes by default

In [ ]:
client = Client(threads_per_worker=1)
client

### Define plot function

In [ ]:
blue_marble = plt.imread('/g/data/ik11/grids/BlueMarble.tiff')
blue_marble_extent = (-180, 180, -90, 90)

In [ ]:
#cb - Jan 25 - tweaked to make it easier to interpret what's going on (to me at least!)
def plot(dat, **kwargs):
    levels=kwargs['levels']
    vmin=kwargs['vmin']
    vmax=kwargs['vmax']
    cmap=kwargs['cmap']
    pnum = kwargs.pop('pnum', '1')
    title = kwargs.pop('title', dat.attrs.get('long_name', ''))
    #not used in this notebook!
    #if title is None:
    #    title = dat.attrs['long_name']
    fig = plt.figure(figsize=(12, 6))
    ax = plt.axes(projection=ccrs.Robinson(central_longitude=-100))
    dat.plot.contourf(
        ax=ax,
        transform=ccrs.PlateCarree(),
        cbar_kwargs={"label": "\n".join(wrap(f"{dat.attrs['long_name']} [{dat.attrs['units']}]", 45)),
                     "fraction": 0.03, "aspect": 15, "shrink": 0.7},
        **kwargs
    )
    
    # Add blue marble land:
    ax.imshow(
        blue_marble, extent=blue_marble_extent, transform=ccrs.PlateCarree(), origin="upper"
    )
    
    plt.title(title)
    mkmd.savefig(fig, "Mixed Layer Depth Max", "ACCESS-OM3 maximum mixed layer depth. [GitHub issue: Time-max mixed layer depth maps](https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/38)")


In [ ]:
#dat=2
#plot(dat,
#        levels=51,
#        vmin=0,
#        vmax=None,
        # extend="max",
#        cmap='viridis',
#        pnum='0',
#       title="meh"
#        )

### Load and plot data from ACCESS-OM3

In [ ]:
exptname = os.path.basename(os.path.dirname(esm_file))

datastore = intake.open_esm_datastore(
    esm_file,
    columns_with_iterables=[
        "variable",
        "variable_long_name",
        "variable_standard_name",
        "variable_cell_methods",
        "variable_units"
    ]
)

In [ ]:
exptname

In [ ]:
#geolon = datastore.search(variable="geolon").to_dask().geolon
#geolat = datastore.search(variable="geolat").to_dask().geolat

#migrating from 25km-iaf-test-for-AK-expt-7df5ef4c to MC_25km_jra_iaf-1.0-beta-5165c0f8
_path = datastore.search(filename=".*static.*", variable=["geolon", "geolat"]).df.loc[0, "path"]
geolon = datastore.search(filename=".*static.*", variable="geolon", path=_path).to_dask().geolon
geolat = datastore.search(filename=".*static.*", variable="geolat", path=_path).to_dask().geolat


In [ ]:
variable = "mlotst_max"
model_all = datastore.search(variable=variable, frequency="1mon").to_dask(
    xarray_open_kwargs = dict(
        chunks={"time": -1},
        decode_timedelta=True
    ),
    xarray_combine_by_coords_kwargs=dict(
        compat="override",
        data_vars="minimal",
        coords="minimal"
    )
)[variable].cf.assign_coords({ "longitude": geolon, "latitude": geolat })

In [ ]:
#migrating from 25km-iaf-test-for-AK-expt-7df5ef4c to MC_25km_jra_iaf-1.0-beta-5165c0f8

##### ONLY NEEDED FOR 25km-iaf-test-for-AK-expt-7df5ef4c !!!!!!!!!!!!!!!!!!!!!!!!!!!!!

# omit latitudes with grid bug https://github.com/ACCESS-NRI/ocean_model_grid_generator/issues/7
#model_all = model_all.isel(yh=slice(10, None))
#geolon = geolon.isel(yh=slice(10, None))
#geolat = geolat.isel(yh=slice(10, None))

In [ ]:
# for IAF
model_all = model_all.convert_calendar("proleptic_gregorian", use_cftime=True)

In [ ]:
model_all.time.values[0] # initial date in data

In [ ]:
model_all.time.values[-1] # final date in data

In [ ]:
# set time range

# timerange = slice(cftime.DatetimeNoLeap(1942, 1, 1, 0, 0, 0, 0),
#                   cftime.DatetimeNoLeap(1952, 1, 1, 0, 0, 0, 0))
# timerange = slice(None, None)
# datestop = model_all.time.values[-1] # final date in data
# datelist = list(cftime.to_tuple(datestop))
# datelist[0] -= 10  # last 10 years
# datestart = cftime.datetime(*datelist, calendar=datestop.calendar)
# timerange = slice(datestart, datestop)
timeranges = [ # see https://github.com/ACCESS-Community-Hub/access-om3-paper-1/issues/34#issuecomment-3300900496
    slice(cftime.DatetimeProlepticGregorian(1958, 1, 1, 0, 0, 0, 0),
          cftime.DatetimeProlepticGregorian(1986, 1, 1, 0, 0, 0, 0)),
    slice(cftime.DatetimeProlepticGregorian(1986, 1, 1, 0, 0, 0, 0),
          cftime.DatetimeProlepticGregorian(2004, 1, 1, 0, 0, 0, 0)),
    slice(cftime.DatetimeProlepticGregorian(2004, 1, 1, 0, 0, 0, 0),
          cftime.DatetimeProlepticGregorian(2021, 1, 1, 0, 0, 0, 0)),
]

In [ ]:
%%time
pnumber='0'
for timerange in timeranges:
    model_sliced = model_all.sel(time=timerange)
    model = model_sliced.max('time').load()
    plot(model,
        levels=51,
        vmin=0,
        vmax=None,
        # extend="max",
        cmap='viridis',
        pnum=pnumber,
        title=f"{model.attrs['long_name']} {model_sliced.time.values[0].strftime('%Y-%m-%d')} - {model_sliced.time.values[-1].strftime('%Y-%m-%d')} maximum in {exptname}"
        )
    pnumber=str(int(pnumber)+1)

In [ ]:
%%time
pnumber='10'
for timerange in timeranges:
    model_sliced = model_all.sel(time=timerange)
    model = model_sliced.max('time').load()
    plot(model,
        levels=51,
        vmin=0,
        vmax=1200,
        extend="max",
        cmap='viridis',
        title=f"{model.attrs['long_name']} {model_sliced.time.values[0].strftime('%Y-%m-%d')} - {model_sliced.time.values[-1].strftime('%Y-%m-%d')} maximum in {exptname}",
        pnum=pnumber
        )
    pnumber=str(int(pnumber)+1)

### Load and plot obs data from DeBoyer Montegut (2023)
https://doi.org/10.17882/91774

In [ ]:
MLDobs = xr.open_dataset('/g/data/av17/access-nri/OM3/MLD-DeBoyerMontegut2023/mld_dr003_ref10m_v2023.nc')['mld_dr003']
MLDobs.attrs['units'] = MLDobs.attrs['unit']  # fix so plot works

# TODO: append copy of westernmost data to eastern end to avoid gap in plot

In [ ]:
MLDobs_max = MLDobs.max('time').load()

In [ ]:
plot(MLDobs_max,
    levels=51,
    vmin=0,
    vmax=None,
    # extend="max",
    cmap='viridis',
    title=f"Observed maximum mixed layer depth (DeBoyer Montegut, 2023)",
    pnum='50')

In [ ]:
client.close()